In [0]:
dim_table = dbutils.widgets.get("dim_table")
raw_table = dbutils.widgets.get("raw_table")
date_table = dbutils.widgets.get("date_table")
payer_table = dbutils.widgets.get("payer_table")
office_table = dbutils.widgets.get("office_table")
source_system_table = dbutils.widgets.get("source_system_table")
client_table = dbutils.widgets.get("client_table")

In [0]:
display(
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW dim_src AS
  SELECT
    CASE
      WHEN (OL.ar_0_90 + OL.ar_91_180 + OL.ar_181_270 + OL.ar_271_plus) < 0 THEN 'Credit'
      ELSE 'Debit'
    END AS invoice_balance_type,
    S.SourceSystemKey        AS source_system_key,
    O.OfficeKey              AS office_key,
    C.ClientKey              AS client_key,
    P.PayerKey               AS payor_key,
    DE.DateKey               AS date_entered_key,
    ID.DateKey               AS invoice_date_key,
    FV.DateKey               AS first_visit_date_key,
    LV.DateKey               AS last_visit_date_key,
    LP.CalendarDate          AS last_payment_date,
    OL.payor_type_code       AS payor_type_code,
    OL.collector_name        AS collector_name,
    OL.total_days_serviced   AS total_days_of_service,
    OL.invno                 AS invoice_number,
    OL.net_revenue           AS net_revenue,
    OL.origbill              AS orig_bill,
    OL.account_balance       AS account_balance,
    OL.total_payments        AS total_payments,
    OL.total_adjustments     AS total_adjustments,
    OL.ar_0_90               AS ar_0_90,
    OL.ar_91_180             AS ar_91_180,
    OL.ar_181_270            AS ar_181_270,
    OL.ar_271_plus           AS ar_271_plus,
    OL.npi_number            AS npi_number,
    OL._load_timestamp       AS _load_timestamp,
    OL._file_name            AS _file_name
  FROM {raw_table} OL
  LEFT JOIN {source_system_table} S
    ON S.SourceSystemName = OL.source_system
  LEFT JOIN {office_table} O
    ON O.OfficeNumber = OL.Office
  LEFT JOIN {payer_table} P
    ON P.PayerID = OL.BILLTO
  LEFT JOIN (
      SELECT MAX(ClientKey) AS ClientKey, SourceSystemId, OfficeNumber
      FROM {client_table}
      GROUP BY SourceSystemId, OfficeNumber
  ) C
    ON C.SourceSystemId = OL.CLIENTNO
    AND C.OfficeNumber  = OL.office
  LEFT JOIN {date_table} DE
  ON DE.CalendarDate = OL.date_entered
  LEFT JOIN {date_table} ID
    ON ID.CalendarDate = OL.invoice_date
  LEFT JOIN {date_table} FV
    ON FV.CalendarDate = OL.first_visit_date
  LEFT JOIN {date_table} LV
    ON LV.CalendarDate = OL.last_visit_date
  LEFT JOIN {date_table} LP
    ON LP.CalendarDate = OL.last_payment_date
""")
)

In [0]:
display(
spark.sql(f"""
MERGE INTO {dim_table} tgt
USING (
    SELECT *
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY
                    invoice_balance_type,
                    payor_type_code,
                    collector_name,
                    total_days_of_service,
                    invoice_number,
                    net_revenue,
                    orig_bill,
                    account_balance,
                    total_payments,
                    total_adjustments,
                    ar_0_90,
                    ar_91_180,
                    ar_181_270,
                    ar_271_plus,
                    npi_number,
                    date_entered_key
                ORDER BY _load_timestamp DESC
            ) AS rn
        FROM dim_src
    ) sub
    WHERE rn = 1
) src
ON tgt.invoice_balance_type = src.invoice_balance_type
AND tgt.payor_type_code        = src.payor_type_code
AND tgt.collector_name        = src.collector_name
AND tgt.total_days_of_service = src.total_days_of_service
AND tgt.invoice_number        = src.invoice_number
AND tgt.net_revenue           = src.net_revenue
AND tgt.orig_bill             = src.orig_bill
AND tgt.account_balance       = src.account_balance
AND tgt.total_payments        = src.total_payments
AND tgt.total_adjustments     = src.total_adjustments
AND tgt.ar_0_90               = src.ar_0_90
AND tgt.ar_91_180             = src.ar_91_180
AND tgt.ar_181_270            = src.ar_181_270
AND tgt.ar_271_plus           = src.ar_271_plus
AND tgt.npi_number            = src.npi_number
AND tgt.date_entered_key  = src.date_entered_key

WHEN NOT MATCHED THEN
INSERT (
    invoice_balance_type,
    source_system_key,
    office_key,
    client_key,
    payor_key,
    date_entered_key,
    invoice_date_key,
    first_visit_date_key,
    last_visit_date_key,
    last_payment_date,
    payor_type_code,
    collector_name,
    total_days_of_service,
    invoice_number,
    net_revenue,
    orig_bill,
    account_balance,
    total_payments,
    total_adjustments,
    ar_0_90,
    ar_91_180,
    ar_181_270,
    ar_271_plus,
    npi_number,
    _load_timestamp,
    _file_name 
)
VALUES (
    src.invoice_balance_type,
    src.source_system_key,
    src.office_key,
    src.client_key,
    src.payor_key,
    src.date_entered_key,
    src.invoice_date_key,
    src.first_visit_date_key,
    src.last_visit_date_key,
    src.last_payment_date,
    src.payor_type_code,
    src.collector_name,
    src.total_days_of_service,
    src.invoice_number,
    src.net_revenue,
    src.orig_bill,
    src.account_balance,
    src.total_payments,
    src.total_adjustments,
    src.ar_0_90,
    src.ar_91_180,
    src.ar_181_270,
    src.ar_271_plus,
    src.npi_number,
    src._load_timestamp,
    src._file_name
)
""")
)